# Notebook 12: Final Model Evaluation, TTA Sweep, and Cross-Domain Generalization Audit

**Repository**: `DLbasedRoadCracks/road-pci-intelligence`  
**Purpose**: Official evaluation of the frozen segmentation model (**Run B: Naive Multi-Task Loss, Seed 42, Epoch 30**).

### Protocol:
1. **Housekeeping & Verification**: Load frozen weights dataset `vedshah04/road-pci-seg-final-weights`, verify SHA-256 digest (`9269c5edb6e5d37b359bd3337fbd83c60901cd2b241f9ef789e9116fee400ef7`), and verify 24,436,514 trainable parameters.
2. **Phase 1: Test-Time Augmentation (TTA) Sweep on Validation Split**:
   - Compare No-TTA vs. Flip-TTA (average of original, h-flip, v-flip; sigmoid then mean).
   - Pre-registered decision rule: Use Flip-TTA as official setting only if mean (crack500, pothole) Val Dice improves by $\ge 0.005$; otherwise No-TTA.
3. **Phase 2: Official Test Split Evaluation**:
   - Evaluate all test splits (**CRACK500: 72**, **Pothole: 60**, **Road Crack: 68**) with both settings in a single run at threshold 0.5.
   - Report Dataset IoU, Dataset Dice, Mean Img Dice; for Road Crack also Non-Empty Dice and empty-clean count. Mark official setting.
4. **Phase 3: Out-of-Domain RDD Test Split Check**:
   - Evaluate the RDD2022ES Test Split (**2,560 images**) with the final model. Document as an out-of-domain limitation.
5. **Phase 4: Visualizations**:
   - Save 8 overlays per source from the test sets for the README (including the 2 lowest-Dice failure cases per source).


In [ ]:
# ==============================================================================
# 1. REPOSITORY SETUP AND ENVIRONMENT INITIALIZATION
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

# 1. Install required packages
print("Installing segmentation-models-pytorch and albumentations...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8"], check=True)

# 2. Clone or set up repo path
repo_dir = Path('/kaggle/working/repo')
if not repo_dir.exists():
    print("Cloning repository from GitHub...")
    subprocess.run(["git", "clone", "https://github.com/skilloutofbounds/road-pci-intelligence.git", str(repo_dir)], check=True)
else:
    print(f"Repo already exists at {repo_dir}")
    subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import time
import hashlib
import json
import random

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast
import torchvision
import albumentations as A
import segmentation_models_pytorch as smp

from src.build_manifest import DATASET_SPECS, resolve_dataset_root
from src.datasets import RoadDistressDataset
from src.metrics import DistressMetricTracker, compute_binary_metrics
from src.data_rules import load_standardized_mask

print("=" * 80)
print("ENVIRONMENT INFORMATION:")
print(f"  • Python:       {sys.version.split()[0]}")
print(f"  • PyTorch:      {torch.__version__}")
print(f"  • Torchvision:  {torchvision.__version__}")
print(f"  • CUDA Device:  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  • SMP Version:  {smp.__version__}")
print("=" * 80)


In [ ]:
# ==============================================================================
# 2. LOCATE FINAL WEIGHTS & VERIFY INTEGRITY (SHA-256 AND PARAM COUNT)
# ==============================================================================
weights_candidates = [
    Path('/kaggle/input/road-pci-seg-final-weights/best.pt'),
    Path('/kaggle/input/datasets/vedshah04/road-pci-seg-final-weights/best.pt'),
] + list(Path('/kaggle/input').rglob('best.pt'))

weights_path = None
for cand in weights_candidates:
    if cand.exists() and cand.is_file():
        weights_path = cand.resolve()
        break

if weights_path is None:
    raise FileNotFoundError("CRITICAL: best.pt weights file could not be found in /kaggle/input!")

print(f"Found final weights at: {weights_path} ({weights_path.stat().st_size:,} bytes)")

# Compute SHA-256
sha256 = hashlib.sha256()
with open(weights_path, 'rb') as f:
    while chunk := f.read(1024 * 1024):
        sha256.update(chunk)
weights_digest = sha256.hexdigest()

EXPECTED_DIGEST = "9269c5edb6e5d37b359bd3337fbd83c60901cd2b241f9ef789e9116fee400ef7"
EXPECTED_PARAMS = 24436514

print(f"Computed SHA-256: {weights_digest}")
print(f"Expected SHA-256: {EXPECTED_DIGEST}")
assert weights_digest == EXPECTED_DIGEST, f"FATAL: SHA-256 mismatch! Got {weights_digest}"
print("[PASS] SHA-256 checksum verified perfectly.")

# Build model architecture and load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights=None,
    in_channels=3,
    classes=2,
    activation=None
)

ckpt = torch.load(weights_path, map_location='cpu', weights_only=False)
if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
    state_dict = ckpt['model_state_dict']
    ckpt_epoch = ckpt.get('epoch', 'N/A')
elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
    state_dict = ckpt['state_dict']
    ckpt_epoch = ckpt.get('epoch', 'N/A')
else:
    state_dict = ckpt
    ckpt_epoch = 'N/A'

missing, unexpected = model.load_state_dict(state_dict, strict=True)
assert len(missing) == 0 and len(unexpected) == 0, f"FATAL: Mismatched keys: missing={missing}, unexpected={unexpected}"

total_params = sum(p.numel() for p in model.parameters())
assert total_params == EXPECTED_PARAMS, f"FATAL: Param count mismatch! Got {total_params}"

model = model.to(device)
model.eval()

print(f"Checkpoint Epoch: {ckpt_epoch}")
print(f"Total Parameters: {total_params:,} (Strict match, 0 missing, 0 unexpected)")
print(f"[PASS] Model successfully loaded on {device}")


In [ ]:
# ==============================================================================
# 3. LOCATE DATASETS & INITIALIZE MANIFESTS
# ==============================================================================
manifest_path = repo_dir / 'manifests/segmentation_manifest.csv'
manifest_df = pd.read_csv(manifest_path)
print(f"Loaded segmentation manifest: {len(manifest_df):,} total records")

dataset_roots = {}
for src, meta in DATASET_SPECS.items():
    if src == 'rdd2022es':
        continue
    resolved = resolve_dataset_root(meta['owner'], meta['name'])
    if resolved is None:
        raise FileNotFoundError(f"CRITICAL: Failed to locate dataset root for {src}")
    dataset_roots[src] = resolved
    print(f"  • {src:<12} -> {resolved}")

# Check Val and Test splits
val_df = manifest_df[manifest_df['split'] == 'val']
test_df = manifest_df[manifest_df['split'] == 'test']

print(f"\nSplit counts:")
print(f"  • Val split:  {len(val_df):,} images (C500: {(val_df['source']=='crack500').sum()}, Pot: {(val_df['source']=='pothole').sum()}, RC: {(val_df['source']=='road_crack').sum()})")
print(f"  • Test split: {len(test_df):,} images (C500: {(test_df['source']=='crack500').sum()}, Pot: {(test_df['source']=='pothole').sum()}, RC: {(test_df['source']=='road_crack').sum()})")

val_dataset = RoadDistressDataset(manifest_df=manifest_df, dataset_roots=dataset_roots, split='val', crop_size=512)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

test_dataset = RoadDistressDataset(manifest_df=manifest_df, dataset_roots=dataset_roots, split='test', crop_size=512)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

print(f"[OK] Val DataLoader ({len(val_loader)} batches) and Test DataLoader ({len(test_loader)} batches) ready.")


In [ ]:
# ==============================================================================
# 4. PHASE 1: TEST-TIME AUGMENTATION (TTA) SWEEP ON VALIDATION SPLIT
# ==============================================================================
# Pre-registered rule:
# Compare No-TTA vs. Flip-TTA (average of original, h-flip, v-flip; sigmoid then mean).
# Flip-TTA is adopted as official ONLY IF mean (crack500, pothole) val Dice improves by >= 0.005.

print("=" * 80)
print("EVALUATING VALIDATION SPLIT: NO-TTA vs. FLIP-TTA (Threshold 0.5)")
print("=" * 80)

tracker_no_tta = DistressMetricTracker(threshold=0.5)
tracker_flip_tta = DistressMetricTracker(threshold=0.5)

t0_val = time.time()
with torch.no_grad():
    for idx, (images, targets, _, metas) in enumerate(val_loader):
        images = images.to(device, non_blocking=True)
        src = metas['source'][0]
        unpad_info = (
            int(metas['eval_unpad'][0][0]),
            int(metas['eval_unpad'][1][0]),
            int(metas['eval_unpad'][2][0]),
            int(metas['eval_unpad'][3][0])
        )
        
        # 1. Forward No-TTA
        with autocast(enabled=(device.type == 'cuda')):
            logits_orig = model(images)
            probs_orig = torch.sigmoid(logits_orig)
            
            # 2. Forward Flip-TTA (horizontal & vertical flips)
            # H-flip
            img_h = torch.flip(images, dims=[-1])
            logits_h = model(img_h)
            probs_h = torch.flip(torch.sigmoid(logits_h), dims=[-1])
            
            # V-flip
            img_v = torch.flip(images, dims=[-2])
            logits_v = model(img_v)
            probs_v = torch.flip(torch.sigmoid(logits_v), dims=[-2])
            
            # Sigmoid then mean
            probs_tta = (probs_orig + probs_h + probs_v) / 3.0

        tracker_no_tta.update(src, probs_orig[0], targets[0], eval_unpad=unpad_info)
        tracker_flip_tta.update(src, probs_tta[0], targets[0], eval_unpad=unpad_info)

val_time = time.time() - t0_val
print(f"[OK] Validation evaluation completed in {val_time:.2f}s ({len(val_loader)} images)")

metrics_val_no_tta = tracker_no_tta.compute()
metrics_val_flip_tta = tracker_flip_tta.compute()

# Pre-registered selection metric: mean of crack500 and pothole dataset dice
score_no_tta = (metrics_val_no_tta['crack500_dataset_dice'] + metrics_val_no_tta['pothole_dataset_dice']) / 2.0
score_flip_tta = (metrics_val_flip_tta['crack500_dataset_dice'] + metrics_val_flip_tta['pothole_dataset_dice']) / 2.0
delta_tta = score_flip_tta - score_no_tta

print("\n" + "=" * 90)
print(f"{'Setting':<12} | {'Mean Dice (C500+Pot)':<22} | {'CRACK500 Dice':<16} | {'Pothole Dice':<16} | {'Road Crack Dice':<16}")
print("-" * 90)
print(f"{'No-TTA':<12} | {score_no_tta:22.4f} | {metrics_val_no_tta['crack500_dataset_dice']:16.4f} | {metrics_val_no_tta['pothole_dataset_dice']:16.4f} | {metrics_val_no_tta['road_crack_dataset_dice']:16.4f}")
print(f"{'Flip-TTA':<12} | {score_flip_tta:22.4f} | {metrics_val_flip_tta['crack500_dataset_dice']:16.4f} | {metrics_val_flip_tta['pothole_dataset_dice']:16.4f} | {metrics_val_flip_tta['road_crack_dataset_dice']:16.4f}")
print("-" * 90)
print(f"Delta (Flip-TTA - No-TTA): {delta_tta:+.5f} (Required improvement: >= +0.00500)")
print("=" * 90)

if delta_tta >= 0.005:
    OFFICIAL_SETTING = "Flip-TTA"
    print(f"\n>>> PRE-REGISTERED DECISION: Flip-TTA improved score by {delta_tta:+.5f} >= 0.00500. OFFICIAL SETTING = Flip-TTA <<<\n")
else:
    OFFICIAL_SETTING = "No-TTA"
    print(f"\n>>> PRE-REGISTERED DECISION: Flip-TTA delta {delta_tta:+.5f} < 0.00500. OFFICIAL SETTING = No-TTA <<<\n")


In [ ]:
# ==============================================================================
# 5. PHASE 2: OFFICIAL TEST SPLIT EVALUATION (CRACK500: 72, POTHOLE: 60, ROAD CRACK: 68)
# ==============================================================================
# Evaluates test splits with BOTH settings in a single run.
# Records per-image predictions and Dice for overlay plotting and failure case identification.

print("=" * 80)
print(f"EVALUATING TEST SPLITS (N = {len(test_loader)} images, Threshold 0.5)")
print(f"OFFICIAL PRE-SELECTED SETTING: {OFFICIAL_SETTING}")
print("=" * 80)

tracker_test_no_tta = DistressMetricTracker(threshold=0.5)
tracker_test_flip_tta = DistressMetricTracker(threshold=0.5)

test_records = []

t0_test = time.time()
with torch.no_grad():
    for idx, (images, targets, _, metas) in enumerate(test_loader):
        images = images.to(device, non_blocking=True)
        src = metas['source'][0]
        img_rel_path = metas['image_path'][0]
        unpad_info = (
            int(metas['eval_unpad'][0][0]),
            int(metas['eval_unpad'][1][0]),
            int(metas['eval_unpad'][2][0]),
            int(metas['eval_unpad'][3][0])
        )
        
        with autocast(enabled=(device.type == 'cuda')):
            logits_orig = model(images)
            probs_orig = torch.sigmoid(logits_orig)
            
            img_h = torch.flip(images, dims=[-1])
            logits_h = model(img_h)
            probs_h = torch.flip(torch.sigmoid(logits_h), dims=[-1])
            
            img_v = torch.flip(images, dims=[-2])
            logits_v = model(img_v)
            probs_v = torch.flip(torch.sigmoid(logits_v), dims=[-2])
            
            probs_tta = (probs_orig + probs_h + probs_v) / 3.0

        tracker_test_no_tta.update(src, probs_orig[0], targets[0], eval_unpad=unpad_info)
        tracker_test_flip_tta.update(src, probs_tta[0], targets[0], eval_unpad=unpad_info)

        # Store for visualizations & failure cases using official setting
        ch = 1 if src == 'pothole' else 0
        p_chosen = probs_tta[0, ch] if OFFICIAL_SETTING == 'Flip-TTA' else probs_orig[0, ch]
        t_chosen = targets[0, ch]
        
        y0, x0, h, w = unpad_info
        p_unpad = p_chosen[y0:y0 + h, x0:x0 + w].detach().cpu().numpy()
        t_unpad = t_chosen[y0:y0 + h, x0:x0 + w].detach().cpu().numpy()
        
        p_bin = (p_unpad >= 0.5).astype(np.uint8)
        t_bin = (t_unpad >= 0.5).astype(np.uint8)
        _, _, _, dice_i = compute_binary_metrics(p_bin, t_bin)
        
        full_img_path = dataset_roots[src] / img_rel_path
        test_records.append({
            'source': src,
            'image_rel_path': img_rel_path,
            'full_image_path': str(full_img_path),
            'channel': 'pothole' if src == 'pothole' else 'crack',
            'gt_pixels': int(t_bin.sum()),
            'pred_pixels': int(p_bin.sum()),
            'dice': float(dice_i),
            'gt_mask': t_bin,
            'pred_mask': p_bin
        })

test_time = time.time() - t0_test
print(f"[OK] Test split evaluation completed in {test_time:.2f}s ({len(test_loader)} images)")

metrics_test_no_tta = tracker_test_no_tta.compute()
metrics_test_flip_tta = tracker_test_flip_tta.compute()

# Print Comprehensive Results Table
print("\n" + "=" * 115)
print(f"{'Setting':<16} | {'Source':<12} | {'N Img':<6} | {'Dataset IoU':<12} | {'Dataset Dice':<14} | {'Mean Img Dice':<14} | {'Non-Empty Dice':<16} | {'Empty Clean':<12}")
print("-" * 115)

for set_name, m in [("No-TTA", metrics_test_no_tta), ("Flip-TTA", metrics_test_flip_tta)]:
    tag = f"{set_name} [OFFICIAL]" if set_name == OFFICIAL_SETTING else set_name
    for s in ['crack500', 'pothole', 'road_crack']:
        n_img = m[f'{s}_images']
        ds_iou = m[f'{s}_dataset_iou']
        ds_dice = m[f'{s}_dataset_dice']
        m_dice = m[f'{s}_mean_image_dice']
        ne_dice = m.get(f'{s}_non_empty_mean_image_dice', float('nan'))
        ec_cnt = m.get(f'{s}_empty_clean_images', 'N/A')
        ne_str = f"{ne_dice:.4f} ({m[f'{s}_non_empty_images']})" if s == 'road_crack' else 'N/A'
        ec_str = f"{ec_cnt} / {n_img - m[f'{s}_non_empty_images']}" if s == 'road_crack' else 'N/A'
        print(f"{tag:<16} | {s:<12} | {n_img:<6} | {ds_iou:12.4f} | {ds_dice:14.4f} | {m_dice:14.4f} | {ne_str:<16} | {ec_str:<12}")
    if set_name == "No-TTA":
        print("-" * 115)
print("=" * 115)


In [ ]:
# ==============================================================================
# 6. PHASE 3: RDD2022ES TEST SPLIT CROSS-DOMAIN AUDIT (2,560 IMAGES)
# ==============================================================================
# Evaluates the frozen model on the 2,560 RDD2022ES test split images.
# Classes: 0 longitudinal, 1 longitudinal wide, 2 transverse, 3 transverse wide,
#          4 alligator, 5 alligator sunken, 6 pothole, 7 pothole deep.

RDD_CLASSES = {
    0: 'longitudinal',
    1: 'longitudinal wide',
    2: 'transverse',
    3: 'transverse wide',
    4: 'alligator',
    5: 'alligator sunken',
    6: 'pothole',
    7: 'pothole deep'
}

rdd_manifest_path = repo_dir / 'manifests/rdd2022es_manifest.csv'
if not rdd_manifest_path.exists():
    raise FileNotFoundError("CRITICAL: manifests/rdd2022es_manifest.csv not found!")

rdd_df = pd.read_csv(rdd_manifest_path)
rdd_test_df = rdd_df[rdd_df['split'] == 'test'].reset_index(drop=True)
print(f"Loaded RDD test split: {len(rdd_test_df):,} images")

rdd_candidates = [
    Path('/kaggle/input/datasets/juusos/rdd2022es'),
    Path('/kaggle/input/rdd2022es'),
    Path('/kaggle/input/juusos/rdd2022es'),
    Path('data/rdd2022es')
]
rdd_root = None
for c in rdd_candidates:
    if c.exists() and c.is_dir():
        rdd_root = c.resolve()
        break
if rdd_root is None:
    for d in Path('/kaggle/input').rglob('rdd2022es'):
        if d.is_dir():
            rdd_root = d.resolve()
            break
if rdd_root is None:
    rdd_root = resolve_dataset_root('juusos', 'rdd2022es')

if rdd_root is None:
    raise FileNotFoundError("CRITICAL: Failed to locate juusos/rdd2022es dataset root")
print(f"RDD Root resolved: {rdd_root}")

# Helper to parse label boxes
def parse_yolo_labels(lbl_path, img_w, img_h):
    if not lbl_path.exists():
        return []
    boxes = []
    with open(lbl_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cid = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                xmin = max(0, int((xc - bw / 2.0) * img_w))
                xmax = min(img_w, int((xc + bw / 2.0) * img_w))
                ymin = max(0, int((yc - bh / 2.0) * img_h))
                ymax = min(img_h, int((yc + bh / 2.0) * img_h))
                boxes.append({
                    'class_id': cid,
                    'box': (ymin, ymax, xmin, xmax),
                    'area': (ymax - ymin) * (xmax - xmin)
                })
    return boxes

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

@torch.no_grad()
def infer_rdd_image(img_rgb):
    h, w = img_rgb.shape[:2]
    new_h = ((h + 31) // 32) * 32
    new_w = ((w + 31) // 32) * 32
    pad_h = new_h - h
    pad_w = new_w - w
    img_padded = cv2.copyMakeBorder(img_rgb, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
    norm = ((img_padded.astype(np.float32) / 255.0) - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.from_numpy(norm).permute(2, 0, 1).unsqueeze(0).float().to(device)
    with autocast(enabled=(device.type == 'cuda')):
        probs = torch.sigmoid(model(tensor))[0, :, :h, :w].cpu().numpy()
    return probs[0], probs[1]

print("\n" + "=" * 80)
print(f"RUNNING RDD TEST SPLIT INFERENCE ({len(rdd_test_df):,} images)...")
print("=" * 80)

t0_rdd = time.time()
rdd_results = []

for idx, row in rdd_test_df.iterrows():
    img_p = rdd_root / row['image_path']
    txt_p = rdd_root / row['mask_path_or_label_path'] if pd.notna(row['mask_path_or_label_path']) else Path('')
    
    with Image.open(img_p) as pil:
        img_rgb = np.array(pil.convert('RGB'))
    h, w = img_rgb.shape[:2]
    tot_pix = h * w
    
    boxes = parse_yolo_labels(txt_p, w, h)
    c_boxes = [b for b in boxes if b['class_id'] in range(6)]
    p_boxes = [b for b in boxes if b['class_id'] in [6, 7]]
    
    p_c, p_p = infer_rdd_image(img_rgb)
    pred_crk = (p_c >= 0.5)
    pred_pot = (p_p >= 0.5)
    
    crk_fg_frac = float(pred_crk.sum()) / tot_pix
    pot_fg_frac = float(pred_pot.sum()) / tot_pix
    
    rdd_results.append({
        'image_path': str(img_p),
        'boxes': boxes,
        'crack_boxes': c_boxes,
        'pothole_boxes': p_boxes,
        'crack_fg_frac': crk_fg_frac,
        'pothole_fg_frac': pot_fg_frac,
        'pred_crack': pred_crk,
        'pred_pothole': pred_pot
    })
    
    if (idx + 1) % 500 == 0 or (idx + 1) == len(rdd_test_df):
        print(f"  Processed {idx + 1:,} / {len(rdd_test_df):,} images ({(idx + 1)/len(rdd_test_df)*100:.1f}%) in {time.time() - t0_rdd:.1f}s")

print(f"[OK] RDD Test inference completed in {time.time() - t0_rdd:.2f}s ({len(rdd_results)} images)")

# Helper to compute box hit rate and pixel coverage
def compute_rdd_box_metrics(records, box_key, pred_key, cid_filter=None):
    n_images = 0
    n_boxes = 0
    hit_images = 0
    box_fractions = []
    
    for r in records:
        b_list = r[box_key]
        if cid_filter is not None:
            b_list = [b for b in b_list if b['class_id'] == cid_filter]
        if not b_list:
            continue
            
        n_images += 1
        n_boxes += len(b_list)
        pred_map = r[pred_key]
        
        img_hit = False
        for b in b_list:
            ymin, ymax, xmin, xmax = b['box']
            b_area = (ymax - ymin) * (xmax - xmin)
            if b_area == 0:
                continue
            pred_in_box = pred_map[ymin:ymax, xmin:xmax].sum()
            frac = float(pred_in_box) / b_area
            box_fractions.append(frac)
            if frac >= 0.05:
                img_hit = True
                
        if img_hit:
            hit_images += 1
            
    hit_rate = (hit_images / n_images * 100.0) if n_images > 0 else 0.0
    mean_box_frac = (np.mean(box_fractions) * 100.0) if box_fractions else 0.0
    return n_images, n_boxes, hit_rate, mean_box_frac

# Part A: Zero-Box Images
clean_rdd = [r for r in rdd_results if len(r['boxes']) == 0]
crk_cands = [r for r in rdd_results if len(r['crack_boxes']) > 0]
pot_cands = [r for r in rdd_results if len(r['pothole_boxes']) > 0]

print("\n" + "=" * 100)
print(f"A. ZERO-BOX IMAGES EVALUATION (N = {len(clean_rdd):,} completely clean images)")
print("=" * 100)
print(f"{'Channel':<16} | {'Flag > 0.1%':<18} | {'Flag > 0.5%':<18} | {'Flag > 1.0%':<18} | {'Mean FG Fraction':<16}")
print("-" * 100)

crk_fracs_clean = np.array([r['crack_fg_frac'] for r in clean_rdd])
pot_fracs_clean = np.array([r['pothole_fg_frac'] for r in clean_rdd])

print(f"{'Crack':<16} | {np.mean(crk_fracs_clean > 0.001)*100:6.2f}% ({np.sum(crk_fracs_clean > 0.001):4d}) | {np.mean(crk_fracs_clean > 0.005)*100:6.2f}% ({np.sum(crk_fracs_clean > 0.005):4d}) | {np.mean(crk_fracs_clean > 0.010)*100:6.2f}% ({np.sum(crk_fracs_clean > 0.010):4d}) | {np.mean(crk_fracs_clean)*100:6.4f}%")
print(f"{'Pothole':<16} | {np.mean(pot_fracs_clean > 0.001)*100:6.2f}% ({np.sum(pot_fracs_clean > 0.001):4d}) | {np.mean(pot_fracs_clean > 0.005)*100:6.2f}% ({np.sum(pot_fracs_clean > 0.005):4d}) | {np.mean(pot_fracs_clean > 0.010)*100:6.2f}% ({np.sum(pot_fracs_clean > 0.010):4d}) | {np.mean(pot_fracs_clean)*100:6.4f}%")
print("=" * 100)

# Part B: Box Coverage and Hit Rates
print("\n" + "=" * 105)
print("B. DISTRESS BOX HIT RATE (>= 5% Box Pixels Predicted) & BOX PIXEL COVERAGE")
print("=" * 105)
print(f"{'Target Distresses':<28} | {'N Images':<10} | {'N Boxes':<10} | {'Image Hit Rate (>= 5%)':<24} | {'Mean Box Pixel Frac':<20}")
print("-" * 105)

n_i, n_b, hr, mbf = compute_rdd_box_metrics(pot_cands, 'pothole_boxes', 'pred_pothole')
print(f"{'All Potholes (Classes 6-7)':<28} | {n_i:<10} | {n_b:<10} | {hr:6.2f}% ({int(round(hr*n_i/100)):3d}/{n_i}) | {mbf:6.2f}%")
for cid in [6, 7]:
    n_ic, n_bc, hrc, mbfc = compute_rdd_box_metrics(pot_cands, 'pothole_boxes', 'pred_pothole', cid_filter=cid)
    c_label = f"  Class {cid} ({RDD_CLASSES[cid]})"
    print(f"{c_label:<28} | {n_ic:<10} | {n_bc:<10} | {hrc:6.2f}% ({int(round(hrc*n_ic/100)):3d}/{n_ic}) | {mbfc:6.2f}%")

print("-" * 105)
n_i, n_b, hr, mbf = compute_rdd_box_metrics(crk_cands, 'crack_boxes', 'pred_crack')
print(f"{'All Cracks (Classes 0-5)':<28} | {n_i:<10} | {n_b:<10} | {hr:6.2f}% ({int(round(hr*n_i/100)):3d}/{n_i}) | {mbf:6.2f}%")
for cid in range(6):
    n_ic, n_bc, hrc, mbfc = compute_rdd_box_metrics(crk_cands, 'crack_boxes', 'pred_crack', cid_filter=cid)
    c_label = f"  Class {cid} ({RDD_CLASSES[cid]})"
    print(f"{c_label:<28} | {n_ic:<10} | {n_bc:<10} | {hrc:6.2f}% ({int(round(hrc*n_ic/100)):3d}/{n_ic}) | {mbfc:6.2f}%")
print("=" * 105)

# Part C: Cross-Channel Audit
only_crk = [r for r in rdd_results if len(r['crack_boxes']) > 0 and len(r['pothole_boxes']) == 0]
only_pot = [r for r in rdd_results if len(r['pothole_boxes']) > 0 and len(r['crack_boxes']) == 0]

print("\n" + "=" * 105)
print("C. CROSS-CHANNEL AUDIT")
print(f"  • Images with ONLY Crack boxes:   {len(only_crk):,} images")
print(f"  • Images with ONLY Pothole boxes: {len(only_pot):,} images")
print("=" * 105)
print(f"{'Condition':<25} | {'Evaluated Channel':<18} | {'Flag > 0.1%':<18} | {'Flag > 0.5%':<18} | {'Mean FG Fraction':<16}")
print("-" * 105)

pot_fracs_crk_only = np.array([r['pothole_fg_frac'] for r in only_crk])
crk_fracs_pot_only = np.array([r['crack_fg_frac'] for r in only_pot])

print(f"{'Only Crack Boxes':<25} | {'Pothole':<18} | {np.mean(pot_fracs_crk_only > 0.001)*100:6.2f}% ({np.sum(pot_fracs_crk_only > 0.001):4d}) | {np.mean(pot_fracs_crk_only > 0.005)*100:6.2f}% ({np.sum(pot_fracs_crk_only > 0.005):4d}) | {np.mean(pot_fracs_crk_only)*100:6.4f}%")
print(f"{'Only Pothole Boxes':<25} | {'Crack':<18} | {np.mean(crk_fracs_pot_only > 0.001)*100:6.2f}% ({np.sum(crk_fracs_pot_only > 0.001):4d}) | {np.mean(crk_fracs_pot_only > 0.005)*100:6.2f}% ({np.sum(crk_fracs_pot_only > 0.005):4d}) | {np.mean(crk_fracs_pot_only)*100:6.4f}%")
print("=" * 105)

print("\n[DISCLAIMER] The above RDD2022ES metrics represent an out-of-domain failure mode analysis documenting dashcam perspective limitations, NOT a competitive performance claim.")


In [ ]:
# ==============================================================================
# 7. PHASE 4: GENERATE 8 TEST OVERLAYS PER SOURCE (INCLUDING 2 FAILURE CASES)
# ==============================================================================
# Generates 8-image 3-column figures (Original Image | Ground Truth | Prediction)
# For each source: 2 lowest-Dice test images (failure cases) + 6 representative samples.

overlay_palette = {
    'crack': [230, 20, 20],      # Red
    'pothole': [20, 100, 240]    # Blue
}

def make_overlay(img_rgb, mask, color, alpha=0.5):
    overlay = img_rgb.copy()
    fg = (mask > 0)
    overlay[fg] = (overlay[fg].astype(float) * (1 - alpha) + np.array(color) * alpha).astype(np.uint8)
    return overlay

for src in ['crack500', 'pothole', 'road_crack']:
    src_recs = [r for r in test_records if r['source'] == src]
    if len(src_recs) == 0:
        continue
        
    # Sort ascending by dice score
    src_recs.sort(key=lambda x: x['dice'])
    
    # 2 lowest-dice failure cases
    fail_cases = src_recs[:2]
    
    # 6 representative samples chosen across the remainder
    remainder = src_recs[2:]
    n_rem = len(remainder)
    if n_rem >= 6:
        # Sample across percentiles [10%, 25%, 50%, 75%, 90%, 99%]
        indices = [int(n_rem * p) for p in [0.10, 0.25, 0.50, 0.70, 0.85, 0.98]]
        # Deduplicate while preserving order
        indices = sorted(list(dict.fromkeys(indices)))
        while len(indices) < 6:
            for cand_idx in range(n_rem):
                if cand_idx not in indices:
                    indices.append(cand_idx)
                    break
            indices.sort()
        chosen_rem = [remainder[i] for i in indices[:6]]
    else:
        chosen_rem = remainder
        
    selected_8 = [('FAILURE CASE 1', fail_cases[0]), ('FAILURE CASE 2', fail_cases[1])]
    for i, r in enumerate(chosen_rem):
        selected_8.append((f'SAMPLE {i+1}', r))
        
    fig, axes = plt.subplots(8, 3, figsize=(15, 28))
    plt.subplots_adjust(top=0.96, bottom=0.02, hspace=0.35, wspace=0.08)
    
    fig.suptitle(f"Test Split Visual Overlays: {src.upper()} (8 Images, Including 2 Failure Cases)", fontsize=14, fontweight='bold', y=0.98)
    
    for row_idx, (cat_label, rec) in enumerate(selected_8):
        with Image.open(rec['full_image_path']) as pil:
            img_rgb = np.array(pil.convert('RGB'))
            
        # Ensure image matches mask dimensions (handles CRACK500 downscale factor 0.5)
        if img_rgb.shape[:2] != rec['gt_mask'].shape[:2]:
            img_rgb = cv2.resize(img_rgb, (rec['gt_mask'].shape[1], rec['gt_mask'].shape[0]), interpolation=cv2.INTER_LINEAR)
            
        color = overlay_palette[rec['channel']]
        gt_overlay = make_overlay(img_rgb, rec['gt_mask'], color, alpha=0.55)
        pred_overlay = make_overlay(img_rgb, rec['pred_mask'], color, alpha=0.55)
        
        stem = Path(rec['image_rel_path']).name
        if len(stem) > 24:
            stem = stem[:21] + '...'
            
        # Col 0: Original
        axes[row_idx, 0].imshow(img_rgb)
        axes[row_idx, 0].set_title(f"[{cat_label}] {stem}\nOriginal Image ({img_rgb.shape[1]}x{img_rgb.shape[0]})", fontsize=8.5, fontweight='bold')
        axes[row_idx, 0].axis('off')
        
        # Col 1: Ground Truth
        axes[row_idx, 1].imshow(gt_overlay)
        axes[row_idx, 1].set_title(f"Ground Truth ({rec['channel']})\nGT Pixels: {rec['gt_pixels']:,}", fontsize=8.5)
        axes[row_idx, 1].axis('off')
        
        # Col 2: Prediction
        axes[row_idx, 2].imshow(pred_overlay)
        axes[row_idx, 2].set_title(f"Predicted Mask (Thresh 0.5)\nPred Pixels: {rec['pred_pixels']:,} | Dice: {rec['dice']:.4f}", fontsize=8.5, fontweight='bold' if 'FAILURE' in cat_label else 'normal')
        axes[row_idx, 2].axis('off')
        
    out_fig = Path(f'/kaggle/working/test_overlays_{src}.png')
    plt.savefig(out_fig, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"[OK] Saved 8-image overlay figure to {out_fig}")


In [ ]:
# ==============================================================================
# 8. EXPORT TEST EVALUATION RESULTS TO JSON
# ==============================================================================
final_results_payload = {
    'weights_digest': weights_digest,
    'total_params': total_params,
    'official_setting': OFFICIAL_SETTING,
    'val_results': {
        'no_tta': metrics_val_no_tta,
        'flip_tta': metrics_val_flip_tta,
        'score_no_tta': score_no_tta,
        'score_flip_tta': score_flip_tta,
        'delta_tta': delta_tta
    },
    'test_results': {
        'no_tta': metrics_test_no_tta,
        'flip_tta': metrics_test_flip_tta
    },
    'rdd_test_results': {
        'clean_images_evaluated': len(clean_rdd),
        'clean_crack_flag_01': float(np.mean(crk_fracs_clean > 0.001) * 100),
        'clean_crack_flag_05': float(np.mean(crk_fracs_clean > 0.005) * 100),
        'clean_crack_flag_10': float(np.mean(crk_fracs_clean > 0.010) * 100),
        'clean_crack_mean_fg': float(np.mean(crk_fracs_clean) * 100),
        'clean_pothole_flag_01': float(np.mean(pot_fracs_clean > 0.001) * 100),
        'clean_pothole_flag_05': float(np.mean(pot_fracs_clean > 0.005) * 100),
        'clean_pothole_flag_10': float(np.mean(pot_fracs_clean > 0.010) * 100),
        'clean_pothole_mean_fg': float(np.mean(pot_fracs_clean) * 100)
    }
}

out_json = Path('/kaggle/working/final_eval_results.json')
with open(out_json, 'w') as f:
    json.dump(final_results_payload, f, indent=2)

print(f"[OK] Saved final evaluation payload to {out_json}")
print("=" * 80)
print("NOTEBOOK 12 EXECUTION COMPLETE")
print("=" * 80)
